# RL and generate
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1/cookbook/notebooks/rl_and_generate.ipynb)

Fine-tune on isolated IDRs, save the model, and generate new sequences.


## Setup
A GPU runtime is recommended: **Runtime → Change runtime type → GPU**. CPU execution is supported but slower.

Run cells in order. Installation is self-contained; no repository clone or account is needed. If Colab requests a session restart after installation, restart before running the imports. The first model load downloads weights.


In [ ]:
import sys

!"{sys.executable}" -m pip install -q "idiom[cookbook] @ git+https://github.com/rotskoff-group/idiom.git@v1"

In [1]:
import gc
import time
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

from idiom import IDiom
from idiom.data.records import Record
from idiom.utils.notebook_helpers import (
    check_context,
    example_file,
    isolated,
    load_inputs,
    save_run,
    split_records,
    train_and_save,
    training_config,
    write_fasta,
)

started = time.perf_counter()

## Settings
Upload a FASTA through Colab’s Files pane, or use the example. Annotated protein headers use `_IDR_start-end` with 1-based, inclusive coordinates.


In [2]:
INPUT_FASTA = None # Default: ProtGPS nucleolus IDRs; otherwise set a FASTA path
INPUT_MODE = "idr" # "idr" for isolated IDRs; "annotated" for full proteins with IDR spans
MAX_RECORDS = 32 # Maximum accepted records; None uses all
MODEL_ID = "jxliu2/idiom-20M" # Pretrained IDiom model or local release directory
DEVICE = "auto" # "cpu" or "cuda"; "auto" uses an available GPU
BATCH_SIZE = 2 # Training and generation batch size
SEED = 0 # Random seed
MAX_STEPS = 20 # Training steps for this demonstration
N = 8 # Number of standalone sequences to generate
MAX_NEW_TOKENS = 96 # Generation limit, including STOP; not a fixed IDR length
OUT_DIR = Path("sft_outputs") / time.strftime("%Y%m%d-%H%M%S") # New timestamped folder per run

In [3]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Load training data
The IDRs are extracted before training. Exact duplicates stay in the same split; the split is not homology-aware.


In [4]:
input_path = INPUT_FASTA or example_file("protgps/nucleolus.fasta", Path("example_inputs"))
records, audit = load_inputs(input_path, INPUT_MODE, MAX_RECORDS)
audit.to_csv(OUT_DIR / "input_audit.csv", index=False)
print("Input audit:", audit.status.value_counts().to_dict())
if not records:
    raise ValueError("No accepted sequences.")

Input audit: {'outside sample limit': 118, 'accepted': 32}


In [5]:
base = IDiom.from_pretrained(MODEL_ID, device=DEVICE)
check_context(records, base.model.cfg.max_seq_len)
del base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
records = isolated(records)
train_records, validation_records = split_records(records, seed=SEED)
train_path = write_fasta(train_records, OUT_DIR / "train.fasta")
validation_path = write_fasta(validation_records, OUT_DIR / "validation.fasta")
print("Train:", len(train_records), "Validation:", len(validation_records))

Train: 26 Validation: 6


## Train
Run a small demonstration job. Training saves a configuration, CSV metrics, a checkpoint, and a reloadable model in `OUT_DIR`. Logging stays local; no account or API key is required.


In [6]:
cfg = training_config("sft", MODEL_ID, OUT_DIR, device=DEVICE, steps=MAX_STEPS, seed=SEED)
cfg.data.train_fasta = str(train_path.resolve())
cfg.data.val_fasta = str(validation_path.resolve())
cfg.data.prompted_prob = 0.0
cfg.data.batch_size = min(BATCH_SIZE, len(train_records))
cfg.data.num_workers = 0
cfg.optim.lr = 1e-5
cfg.optim.warmup_steps = min(5, MAX_STEPS)
release = train_and_save(cfg)

## Inspect training
Inspect the latest logged metrics. Full history is in `training/metrics/`. Blank entries indicate a metric was not logged on that step.


In [7]:
metrics_path = next((OUT_DIR / "training/metrics").glob("version_*/metrics.csv"))
metrics = pd.read_csv(metrics_path)
columns = ["step"] + [name for name in ["train/loss", "val/loss"] if name in metrics]
print(f"Logged {len(metrics)} metric rows; latest values:")
display(metrics[columns].tail(3))

Logged 22 metric rows; latest values:


,step,train/loss,val/loss
19,18,2.705519,NaN
20,19,2.254684,NaN
21,20,NaN,2.51595


## Generate from the saved model


In [8]:
sampling = dict(n=N, batch_size=BATCH_SIZE, seed=SEED, max_new_tokens=MAX_NEW_TOKENS)
adapted_model = IDiom.from_pretrained(release, device=DEVICE)
adapted = adapted_model.generate_unprompted(**sampling)
write_fasta(
    [Record(f"adapted_{i}", s, 0, len(s)) for i, s in enumerate(adapted) if s],
    OUT_DIR / "adapted.fasta",
)
del adapted_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [9]:
table = pd.DataFrame({"sequence": adapted, "length": [len(s) for s in adapted]})
table.to_csv(OUT_DIR / "candidates.csv", index=False)
with pd.option_context("display.max_colwidth", 80):
    display(table.head(3))

,sequence,length
0,DKKEKGNEEEHKQSGGRPHAPPQASSIQVL,30
1,MATANGAQKNSPRNFIPWLVFTLGFSGGYLAYRAFYLKQQVLLHRGGSAPIPAIRGHRAATMSLPDAPGTPTGSDT...,96
2,DDDYEDYEAYRDRSNAYRDKAASKPPKVVKPMQAQSKRYTIHEAARPNQRWAPQQNQPPQQAPSDFRRPQLRQRLM...,96


## Results
`candidates.csv` and `adapted.fasta` contain generated IDRs. `model/` is a reloadable IDiom release. `training/` contains logs and a checkpoint; `validation_metrics.json` records validation loss.

`run.json` records the settings and package versions. Open the output folder in Colab’s **Files** pane to download results. Download them before the runtime ends, or copy them to mounted Drive. Saved notebook previews do not include the exported files.


Saved previews show a CPU example run. Run the cells to create the exported files.

In [10]:
save_run(
    OUT_DIR,
    dict(
        model=MODEL_ID,
        device=DEVICE,
        input=str(input_path),
        input_mode=INPUT_MODE,
        seed=SEED,
        steps=MAX_STEPS,
        sampling=sampling,
    ),
    elapsed=time.perf_counter() - started,
)
print("Results folder:", OUT_DIR)

Results folder: sft_outputs/20260921-134247
